# Collect Dataset - Adaptive Speculative Decoding

**Цель:** Прогнать speculative decoding на HumanEval и GSM8K и собрать per-step метрики:
acceptance rate, KL-divergence, perplexity, entropy, agreement rate, ngram match, confidence, timing.

| Parameter | Value |
|-----------|-------|
| Target    | Qwen/Qwen3-8B (NF4) |
| Draft     | Qwen/Qwen3-0.6B (FP32) |
| Benchmarks | HumanEval, GSM8K |
| Draft length | 5 |


In [1]:
import json
import sys
import os

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.append('/Users/pestrovskiy.e/Adaptive-speculative-decoding/src')

import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm

from core.models.draft_model import DraftModel
from core.models.target_model import TargetModel
from core.decoder.speculative import SpeculativeDecoder, StepResult
from core.translation.vocabulary import CrossVocabTranslator
from core.cache.ngram import NgramCache

from dotenv import load_dotenv
load_dotenv()

HF_TOKEN = os.environ['HF_TOKEN']

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("Device: " + device)
if device == "cuda":
    print("GPU: " + torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("GPU memory: " + str(round(props.total_memory / 1e9, 1)) + " GB")
elif device == "mps":
    print("GPU: Apple Silicon MPS")
    print("GPU memory: shared with system")
else:
    print("GPU: N/A (running on CPU)")


Device: mps
GPU: Apple Silicon MPS
GPU memory: shared with system


In [2]:
CONFIG = {
    "target_model": "Qwen/Qwen3-8B",
    "draft_model": "Qwen/Qwen3-0.6B",
    "device": device,
    "target_dtype": torch.float16,
    "target_load_4bit": True,
    "draft_length": 5,
    "temperature": 1.0,
    "max_new_tokens": 128,
    "max_samples": None,
    "datasets": ["humaneval", "gsm8k"],
    "seed": 42,
    "output_dir": "research_data",
}

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

for k, v in CONFIG.items():
    print("  " + k + ": " + str(v))


  target_model: Qwen/Qwen3-8B
  draft_model: Qwen/Qwen3-0.6B
  device: mps
  target_dtype: torch.float16
  target_load_4bit: True
  draft_length: 5
  temperature: 1.0
  max_new_tokens: 128
  max_samples: None
  datasets: ['humaneval', 'gsm8k']
  seed: 42
  output_dir: research_data


In [ ]:
print("=" * 50)
print("[1/2] Loading models...")
target = TargetModel(
    CONFIG["target_model"],
    device=CONFIG["device"],
    dtype=CONFIG["target_dtype"],
    load_in_4bit=CONFIG["target_load_4bit"],
)
print("  Target: " + CONFIG["target_model"])

draft = DraftModel(
    CONFIG["draft_model"],
    device=CONFIG["device"],
)
print("  Draft: " + CONFIG["draft_model"])

print("=" * 50)
print("[2/2] Building CrossVocabTranslator...")
translator = CrossVocabTranslator.from_tokenizers(
    drafter_tokenizer=draft.tokenizer,
    target_tokenizer=target.tokenizer,
    device=CONFIG["device"],
    drafter_vocab_size=draft.model.config.vocab_size,
    target_vocab_size=target.model.config.vocab_size,
)
print("  Drafter vocab: " + str(translator.rule1.drafter_size))
print("  Target vocab:  " + str(translator.rule1.target_size))
identical = translator.rule1.drafter_size == translator.rule1.target_size and translator.rule1._valid_mask.all()
print("  Vocabs identical: " + str(identical))


[1/2] Loading models...


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [ ]:
from datasets import load_dataset


def load_and_tokenize_dataset(name, target_tokenizer, max_samples=None):
    print("  Loading " + name + "...")
    if name == "gsm8k":
        ds = load_dataset("openai/gsm8k", "main", split="test")
        texts = [ex["question"] for ex in ds]
    elif name == "humaneval":
        ds = load_dataset("openai/openai_humaneval", split="test")
        texts = [ex["prompt"] for ex in ds]
    else:
        raise ValueError("Unknown dataset: " + name)

    if max_samples is not None:
        texts = texts[:max_samples]

    chunk_size = 256
    result = []
    for i in range(0, len(texts), chunk_size):
        chunk = texts[i : i + chunk_size]
        enc = target_tokenizer(chunk, return_tensors="pt", padding=True, truncation=False)
        for j in range(len(chunk)):
            ids = enc.input_ids[j].unsqueeze(0)
            prompt_len = ids.shape[1]
            result.append((ids, prompt_len))

    lens = [p for _, p in result]
    avg = sum(lens) / max(len(lens), 1)
    print("    " + str(len(result)) + " samples, avg prompt len: " + str(round(avg, 1)))
    return result


print("Loading datasets...")
all_examples = []

for ds_name in CONFIG["datasets"]:
    print("[" + ds_name + "]")
    examples = load_and_tokenize_dataset(ds_name, target.tokenizer, CONFIG["max_samples"])
    for ids, plen in examples:
        all_examples.append((ds_name, ids, plen))

print("")
print("Total: " + str(len(all_examples)) + " examples across " + str(CONFIG["datasets"]))


Loading datasets...
[humaneval]
  Loading humaneval...
    164 samples, avg prompt len: 391.0
[gsm8k]
  Loading gsm8k...
    1319 samples, avg prompt len: 147.5

Total: 1483 examples across ['humaneval', 'gsm8k']


In [5]:
import time as _time


class HookedSpeculativeDecoder(SpeculativeDecoder):
    """Subclass of SpeculativeDecoder that calls a callback after each decode step.
    Only overrides _generate_loop - all _decode_step logic is inherited unchanged.
    """

    def __init__(self, *args, step_hook=None, **kwargs):
        super().__init__(*args, step_hook=step_hook, **kwargs)

    def _generate_loop(self, input_ids, max_new_tokens, adaptive_length_fn, distiller, rng=None):
        prompt_len = input_ids.shape[1]
        output = torch.zeros(
            (input_ids.shape[0], prompt_len + max_new_tokens),
            dtype=input_ids.dtype,
            device=input_ids.device,
        )
        output[:, :prompt_len] = input_ids
        pos = prompt_len
        max_consec_zero = 5
        consec_zero = 0
        ctx_list = input_ids[0].tolist()

        for step_idx in range(max_new_tokens):
            new_tokens = pos - prompt_len
            if new_tokens >= max_new_tokens:
                break

            generated = output[:, :pos]
            k = self._choose_draft_length(generated, adaptive_length_fn)

            step_start = _time.perf_counter()
            result = self._decode_step(
                generated, k, ctx_list,
                drafter_ctx=generated,
                distiller=distiller,
                rng=rng,
                step_hook=self._step_hook,
                step_idx=step_idx,
                start_time=step_start,
            )

            self._step_results.append(result)
            self.cache.step()

            budget = max_new_tokens - new_tokens
            emitted = result.accepted_tokens[:budget]
            if emitted:
                for i, tok in enumerate(emitted):
                    output[0, pos + i] = tok
                pos += len(emitted)
                ctx_list.extend(emitted)
                consec_zero = 0
            else:
                consec_zero += 1
                if consec_zero >= max_consec_zero:
                    break

            if pos > prompt_len and self._is_eos(output[0, pos - 1]):
                break

        return output[:, :pos]


class DataCollector:
    # Collects per-step and per-example metrics during speculative decoding.
    # Per-step: k, n_accepted, acceptance_ratio, draft_logprob_sum,
    #   kl_divergence, kl_divergence_mean, target_entropy, draft_entropy,
    #   token_agreement_rate, ngram_match_rate, draft_confidence,
    #   time_per_step_ms, effective_throughput

    def __init__(self, decoder):
        self.decoder = decoder
        self.example_records = []
        self._current_steps = []
        decoder._step_hook = self._on_step
        self._example_start = 0.0

    def _on_step(self,
        draft_tokens_drafter, draft_logits, translated_probs,
        target_logits, draft_tokens_target,
        accepted_count, rejected_at,
        step_idx=None, start_time=None, k=None, ctx_list=None,
        wall_time_ms=None, draft_tokens=None):
        """Callback after each _decode_step. Tensors come as kwargs."""
        n_accepted = accepted_count
        acceptance_ratio = n_accepted / max(k, 1)
        time_per_step_ms = (_time.perf_counter() - start_time) * 1000

        step = {
            "step_idx": step_idx,
            "k": k,
            "n_accepted": n_accepted,
            "acceptance_ratio": acceptance_ratio,
            "rejected_at": rejected_at,
        }

        # draft_logprob_sum: -cross_entropy(draft_logits, draft_tokens_drafter)
        if draft_logits is not None and draft_tokens_drafter:
            try:
                dt_tensor = torch.tensor(draft_tokens_drafter, dtype=torch.long,
                                         device=draft_logits.device)
                logits_2d = draft_logits if draft_logits.dim() == 2 else draft_logits.squeeze(1)
                ce_loss = F.cross_entropy(logits_2d, dt_tensor)
                step["draft_logprob_sum"] = -ce_loss.item()
            except Exception:
                step["draft_logprob_sum"] = 0.0
        else:
            step["draft_logprob_sum"] = 0.0

        # KL(P||Q), entropy at last token position
        if translated_probs is not None and target_logits is not None:
            q = translated_probs
            p_logits = target_logits[:k]
            with torch.no_grad():
                p = F.softmax(p_logits.float(), dim=-1)
                q_safe = q.clamp(min=1e-8)
                p_safe = p.clamp(min=1e-8)
                # KL divergence at last position
                kl_last = (p[-1] * (torch.log(p_safe[-1]) - torch.log(q_safe[-1]))).sum().item()
                # Mean KL across all k positions
                kl_mean = ((p * (torch.log(p_safe) - torch.log(q_safe))).sum(dim=-1).mean()).item()
                # Entropies at last position
                target_entropy = -(p[-1] * torch.log(p_safe[-1])).sum().item()
                draft_entropy = -(q[-1] * torch.log(q_safe[-1])).sum().item()
                # Token agreement: argmax(target_probs) == draft_tokens_target?
                target_argmax = p_logits.argmax(dim=-1).tolist()
                if draft_tokens_target:
                    agreement = sum(1 for a, b in zip(target_argmax, draft_tokens_target)
                                    if a == b) / max(len(draft_tokens_target), 1)
                else:
                    agreement = 0.0
                # Draft confidence: max softmax probability at last position
                draft_confidence = float(p[-1].max().item())
                # N-gram match rate (approx by agreement)
                ngram_match = agreement
            step["kl_divergence"] = kl_last
            step["kl_divergence_mean"] = kl_mean
            step["target_entropy"] = target_entropy
            step["draft_entropy"] = draft_entropy
            step["token_agreement_rate"] = agreement
            step["ngram_match_rate"] = ngram_match
            step["draft_confidence"] = draft_confidence
        else:
            step["kl_divergence"] = 0.0
            step["kl_divergence_mean"] = 0.0
            step["target_entropy"] = 0.0
            step["draft_entropy"] = 0.0
            step["token_agreement_rate"] = acceptance_ratio
            step["ngram_match_rate"] = acceptance_ratio
            step["draft_confidence"] = acceptance_ratio

        step["time_per_step_ms"] = time_per_step_ms
        step["effective_throughput"] = n_accepted / max(time_per_step_ms, 1e-6) * 1000
        self._current_steps.append(step)

    def start_example(self, example_id):
        """Mark the start of a new example."""
        self._current_steps = []
        self._example_start = _time.perf_counter()

    def finalize_example(self, example_id, total_new_tokens, dataset_name):
        """Record example results and return summary."""
        steps = self._current_steps
        total_time_ms = (_time.perf_counter() - self._example_start) * 1000

        total_accepted = sum(s["n_accepted"] for s in steps)
        mean_acceptance = np.mean([s["acceptance_ratio"] for s in steps]) if steps else 0.0
        mean_kl = np.mean([s["kl_divergence"] for s in steps]) if steps else 0.0
        mean_draft_entropy = np.mean([s["draft_entropy"] for s in steps]) if steps else 0.0
        mean_target_entropy = np.mean([s["target_entropy"] for s in steps]) if steps else 0.0
        mean_throughput = total_accepted / max(total_time_ms, 1e-6) * 1000 if steps else 0.0

        record = {
            "example_id": example_id,
            "dataset": dataset_name,
            "n_steps": len(steps),
            "total_accepted": total_accepted,
            "total_new_tokens": total_new_tokens,
            "mean_acceptance_rate": mean_acceptance,
            "mean_kl_div": mean_kl,
            "mean_draft_entropy": mean_draft_entropy,
            "mean_target_entropy": mean_target_entropy,
            "total_time_ms": total_time_ms,
            "mean_throughput": mean_throughput,
            "step_data": steps,
        }
        self.example_records.append(record)
        return record

    def save_to_parquet(self, path, metadata):
        """Save dataset to parquet with metadata."""
        import pandas as pd

        flat_rows = []
        for ex in self.example_records:
            for step in ex["step_data"]:
                row = {}
                for k, v in ex.items():
                    if k != "step_data":
                        row["ex_" + k] = v
                row.update(step)
                flat_rows.append(row)

        df = pd.DataFrame(flat_rows)
        d = os.path.dirname(path)
        if d:
            os.makedirs(d, exist_ok=True)
        df.to_parquet(path, index=False)
        msg = "  Saved: " + path + " (" + str(len(df)) + " step records, " + str(len(self.example_records)) + " examples)"
        print(msg)

    def summary_stats(self):
        """Aggregate statistics across all examples."""
        if not self.example_records:
            return {}

        acc_rates = [r["mean_acceptance_rate"] for r in self.example_records]
        kls = [r["mean_kl_div"] for r in self.example_records]
        ent_d = [r["mean_draft_entropy"] for r in self.example_records]
        ent_t = [r["mean_target_entropy"] for r in self.example_records]
        times = [r["total_time_ms"] for r in self.example_records]
        accepted = [r["total_accepted"] for r in self.example_records]
        steps = [r["n_steps"] for r in self.example_records]

        return {
            "n_examples": len(self.example_records),
            "n_steps_total": sum(steps),
            "total_accepted_tokens": sum(accepted),
            "mean_acceptance_rate": float(np.mean(acc_rates)),
            "std_acceptance_rate": float(np.std(acc_rates)),
            "mean_kl_div": float(np.mean(kls)),
            "mean_draft_entropy": float(np.mean(ent_d)),
            "mean_target_entropy": float(np.mean(ent_t)),
            "mean_total_time_ms": float(np.mean(times)),
            "mean_steps_per_example": float(np.mean(steps)),
            "mean_accepted_per_example": float(np.mean(accepted)),
        }

    def get_step_records(self, example_id):
        """Get all step records for a specific example."""
        for ex in self.example_records:
            if ex["example_id"] == example_id:
                return ex["step_data"]
        return []


In [ ]:
all_results = []

for idx, (ds_name, input_ids, prompt_len) in enumerate(tqdm(all_examples, desc="Collecting dataset")):
    input_ids = input_ids.to(CONFIG["device"])

    decoder = HookedSpeculativeDecoder(
        drafter=draft,
        target=target,
        translator=translator,
        cache=NgramCache(),
        draft_length=CONFIG["draft_length"],
        temperature=CONFIG["temperature"],
    )
    collector = DataCollector(decoder)

    collector.start_example(idx)

    rng = torch.Generator(device=CONFIG["device"])
    rng.manual_seed(CONFIG["seed"] + idx)

    with torch.no_grad():
        output = decoder.generate(
            input_ids,
            max_new_tokens=CONFIG["max_new_tokens"],
            rng=rng,
        )

    new_tokens = output.shape[1] - input_ids.shape[1]
    result = collector.finalize_example(idx, new_tokens, ds_name)
    all_results.append(result)

    del decoder
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

import builtins
builtins._final_collector = collector
builtins._all_results = all_results

print("")
print("Completed " + str(len(all_results)) + " examples.")
stats = collector.summary_stats() if all_results else {}
for k, v in stats.items():
    if isinstance(v, float):
        print("  " + k + ": " + str(round(v, 4)))
    else:
        print("  " + k + ": " + str(v))


In [ ]:
output_dir = CONFIG["output_dir"]
os.makedirs(output_dir, exist_ok=True)

import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = "dataset_" + timestamp + ".parquet"
path = os.path.join(output_dir, filename)

metadata = {
    "target_model": CONFIG["target_model"],
    "draft_model": CONFIG["draft_model"],
    "target_4bit": CONFIG["target_load_4bit"],
    "draft_length": CONFIG["draft_length"],
    "temperature": CONFIG["temperature"],
    "max_new_tokens": CONFIG["max_new_tokens"],
    "datasets": CONFIG["datasets"],
    "seed": CONFIG["seed"],
    "created_at": timestamp,
    "n_examples": len(all_results),
}

collector.save_to_parquet(path, metadata)
print("Full path: " + os.path.abspath(path))


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({
    "figure.figsize": (16, 10),
    "figure.dpi": 100,
    "font.size": 12,
})

# Flatten step data for global views
all_steps = []
for ex in all_results:
    for step in ex["step_data"]:
        row = {}
        for k, v in ex.items():
            if k != "step_data":
                row[k] = v
        row.update(step)
        all_steps.append(row)

df_steps = pd.DataFrame(all_steps) if all_steps else pd.DataFrame()

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle("Dataset Collection - EDA", fontsize=16, y=1.02)

# 1. Accepted tokens vs k (scatter)
ax = axes[0, 0]
if not df_steps.empty:
    ax.scatter(df_steps["k"], df_steps["n_accepted"], alpha=0.5, s=15, edgecolors="none")
    for k_val in sorted(df_steps["k"].unique()):
        mask = df_steps["k"] == k_val
        mean_acc = df_steps.loc[mask, "n_accepted"].mean()
        ax.scatter(k_val, mean_acc, marker="X", color="red", s=80, zorder=5)
        ax.annotate(
            "k=" + str(k_val) + ": " + str(round(mean_acc, 1)),
            xy=(k_val, mean_acc),
            fontsize=9,
        )
    ax.set_xlabel("Draft length (k)")
    ax.set_ylabel("Accepted tokens")
    ax.set_title("Accepted tokens vs draft length")
else:
    ax.text(0.5, 0.5, "No data - run Cell 6 first", ha="center", va="center",
            transform=ax.transAxes)

# 2. Acceptance ratio distribution
ax = axes[0, 1]
if not df_steps.empty:
    ax.hist(df_steps["acceptance_ratio"], bins=50, color="steelblue", edgecolor="white")
    m = df_steps["acceptance_ratio"].mean()
    ax.axvline(m, color="red", linestyle="--", label="mean=" + str(round(m, 3)))
    ax.set_xlabel("Acceptance ratio")
    ax.set_ylabel("Count")
    ax.set_title("Distribution of acceptance ratio")
    ax.legend()
else:
    ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)

# 3. Mean KL divergence per example
ax = axes[1, 0]
if all_results:
    kl_vals = [r["mean_kl_div"] for r in all_results]
    ax.hist(kl_vals, bins=30, color="coral", edgecolor="white")
    m = np.mean(kl_vals)
    ax.axvline(m, color="red", linestyle="--", label="mean=" + str(round(m, 4)))
    ax.set_xlabel("Mean KL divergence")
    ax.set_ylabel("Count")
    ax.set_title("Mean KL divergence per example")
    ax.legend()
else:
    ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)

# 4. Acceptance ratio by dataset
ax = axes[1, 1]
if not df_steps.empty and "dataset" in df_steps.columns:
    for ds in df_steps["dataset"].unique():
        mask = df_steps["dataset"] == ds
        ax.hist(df_steps.loc[mask, "acceptance_ratio"], bins=25, alpha=0.6, label=ds)
    ax.set_xlabel("Acceptance ratio")
    ax.set_ylabel("Count")
    ax.set_title("Acceptance ratio by dataset")
    ax.legend()
else:
    ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)

# 5. Total accepted tokens per example
ax = axes[2, 0]
if all_results:
    accepted_vals = [r["total_accepted"] for r in all_results]
    ax.hist(accepted_vals, bins=30, color="seagreen", edgecolor="white")
    m = np.mean(accepted_vals)
    ax.axvline(m, color="red", linestyle="--", label="mean=" + str(round(m, 1)))
    ax.set_xlabel("Total accepted tokens")
    ax.set_ylabel("Count")
    ax.set_title("Total accepted tokens per example")
    ax.legend()
else:
    ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)

# 6. Time per step
ax = axes[2, 1]
if not df_steps.empty and "time_per_step_ms" in df_steps.columns:
    times = df_steps["time_per_step_ms"].dropna()
    if not times.empty:
        ax.hist(times, bins=50, color="mediumpurple", edgecolor="white")
        m = times.mean()
        ax.axvline(m, color="red", linestyle="--", label="mean=" + str(round(m, 1)) + "ms")
        ax.set_xlabel("Time per step (ms)")
        ax.set_ylabel("Count")
        ax.set_title("Time per step distribution")
        ax.legend()
    else:
        ax.text(0.5, 0.5, "No timing data", ha="center", va="center",
                transform=ax.transAxes)
else:
    ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.show()

# Summary table
sep = "=" * 60
print(sep)
print("  SUMMARY STATISTICS")
print(sep)
stats = collector.summary_stats() if all_results else {}
for k, v in stats.items():
    label = k.replace("_", " ").title()
    if isinstance(v, float):
        print("  " + label.ljust(35) + str(round(v, 4)))
    else:
        print("  " + label.ljust(35) + str(v))
print(sep)
